# 🧠 MedSSL-BHSD: Mean Teacher Semi-Supervised Segmentation trên Dual GPU T4 (Kaggle)

Phân đoạn 5 phân loại xuất huyết não chi tiết (EDH, IPH, IVH, SAH, SDH) + Background trên 2x GPU T4 Kaggle theo phương pháp **Mean Teacher (Tarvainen & Valpola, NeurIPS 2017)** đã được tối ưu hóa cho bài toán mất cân bằng dữ liệu 3D BHSD.

### 🌟 Tính năng & Tối ưu hóa End-to-End:
- **1 Pha huấn luyện duy nhất (End-to-End)**: Huấn luyện liên tục 12,000 iters với mô hình đơn nhánh VNet (`vnet_single`), tiết kiệm ~40% VRAM và thời gian so với các phương pháp 2 pha như PICK.
- **EMA Teacher & Weighted Consistency**: Trọng số Teacher cập nhật liên tục qua Exponential Moving Average (EMA decay 0.99); Consistency Loss tự động downweight background (x0.1) và lọc threshold tin cậy (>0.85) để giải quyết triệt để mất cân bằng voxel xuất huyết.
- **Tối ưu 2 GPU T4 (DDP qua `torchrun`)**: Tự động nhận diện 2x GPU T4, chia đều VRAM (~5.5GB/GPU), tốc độ cao hơn và tận dụng tối đa băng thông NVLink/PCIe.
- **Sliding-Window Validation định kỳ trên Teacher**: Mỗi 500 iters tự động đánh giá mô hình Teacher bằng kỹ thuật cửa sổ trượt 3D (AMP-accelerated) và lưu `best_model.pth`.
- **Checkpoint kép & W&B Artifacts**: Lưu `checkpoint_latest.pth` mỗi 50 iters để chống timeout 12h, đồng thời tích hợp đồng bộ hóa W&B Cloud.
- **Benchmark độc lập (`evaluate.py`)**: Tự động nạp `best_model.pth` để đo FLOPs, Latency, Params và per-class Dice chi tiết.
- **Tự động đóng gói kết quả**: Nén toàn bộ log và checkpoint thành `.tar.gz` sẵn sàng tải về từ tab Output.

### 📦 Bước 1: Cài đặt các thư viện cần thiết

In [ ]:
# Cài đặt thư viện với thanh tiến trình rõ ràng (không dùng -q để theo dõi realtime, tránh cảm giác bị đơ)
!pip install --progress-bar on medpy tensorboardX h5py SimpleITK nibabel wandb pyyaml

### 🚀 Bước 2: Clone & Cập nhật mã nguồn MedSSL-BHSD từ GitHub

In [ ]:
import os
import sys
import subprocess

REPO_NAME = "NguyenVanCongThanh/BHSD_SSL"
GITHUB_TOKEN = ""  # Điền token ghp_xxx nếu repo là Private, nếu Public thì để trống
WORK_DIR = "/kaggle/working/BHSD_SSL"

# Chống treo: Vô hiệu hóa prompt chờ nhập pass ngầm của Git trong non-interactive shell
env = os.environ.copy()
env["GIT_TERMINAL_PROMPT"] = "0"

if GITHUB_TOKEN.strip():
    GITHUB_REPO_URL = f"https://{GITHUB_TOKEN.strip()}@github.com/{REPO_NAME}.git"
else:
    GITHUB_REPO_URL = f"https://github.com/{REPO_NAME}.git"

if not os.path.exists(WORK_DIR):
    print(f"[*] Đang clone repository {REPO_NAME}...", flush=True)
    res = subprocess.run(["git", "clone", GITHUB_REPO_URL, WORK_DIR], env=env, timeout=120)
    if res.returncode != 0:
        raise RuntimeError(f"[X] Không thể clone {REPO_NAME}. Hãy kiểm tra mạng hoặc URL repo!")
else:
    print(f"[*] Đồng bộ mã nguồn mới nhất từ GitHub tại {WORK_DIR}...", flush=True)
    subprocess.run(["git", "-C", WORK_DIR, "fetch", "origin"], env=env, check=True, timeout=60)
    subprocess.run(["git", "-C", WORK_DIR, "checkout", "."], check=True)
    subprocess.run(["git", "-C", WORK_DIR, "clean", "-fd"], check=True)
    res = subprocess.run(["git", "-C", WORK_DIR, "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True, text=True)
    branch = res.stdout.strip() if res.returncode == 0 and res.stdout.strip() else "main"
    subprocess.run(["git", "-C", WORK_DIR, "reset", "--hard", f"origin/{branch}"], check=True)

%cd {WORK_DIR}
!git log -1 --oneline

### 🔍 Bước 3: Định vị Dataset BHSD từ Kaggle Input

In [ ]:
import os
import sys
import glob

INPUT_DIR = "/kaggle/input"
WORKING_DIR = "/kaggle/working"

def find_bhsd_root(base_dir):
    if not os.path.exists(base_dir):
        return None
    for root, dirs, files in os.walk(base_dir):
        if "data" in dirs:
            data_p = os.path.join(root, "data")
            if os.path.exists(data_p) and any(f.endswith('.h5') for f in os.listdir(data_p)):
                return root
        if any(f.endswith('_train.txt') for f in files) and "data" in dirs:
            return root
    return None

print("[*] Đang định vị dataset BHSD...", flush=True)
DATA_ROOT = find_bhsd_root(INPUT_DIR) or find_bhsd_root(WORKING_DIR)

if not DATA_ROOT:
    all_tars = glob.glob(os.path.join(INPUT_DIR, "**", "*.tar*"), recursive=True)
    matching_tars = [p for p in all_tars if any(k in os.path.basename(p).lower() for k in ["bhsd", "h5", "fold"])]
    tar_path = matching_tars[0] if matching_tars else (all_tars[0] if all_tars else None)
    
    if not tar_path:
        raise FileNotFoundError(f"[X] Không tìm thấy dataset BHSD trong {INPUT_DIR}! Hãy bấm '+ Add Input'.")
    
    print(f"[*] Đang giải nén {os.path.basename(tar_path)} vào {WORKING_DIR} (khoảng 1-2 phút)...", flush=True)
    if tar_path.endswith(".tar.gz") or tar_path.endswith(".tgz"):
        !tar -xzf "{tar_path}" -C {WORKING_DIR}
    else:
        !tar -xf "{tar_path}" -C {WORKING_DIR}
    DATA_ROOT = find_bhsd_root(WORKING_DIR)
    assert DATA_ROOT, f"[X] Giải nén xong nhưng không tìm thấy data/ và splits/ trong {WORKING_DIR}!"

print(f"[SUCCESS] Đường dẫn DATA_ROOT: {DATA_ROOT}", flush=True)

# Kiểm tra tính toàn vẹn dữ liệu
splits_dir = os.path.join(DATA_ROOT, "splits") if os.path.exists(os.path.join(DATA_ROOT, "splits")) else DATA_ROOT
data_dir = os.path.join(DATA_ROOT, "data")
h5_files = glob.glob(os.path.join(data_dir, "*.h5"))
print(f"  [+] Tổng số file .h5: {len(h5_files)}", flush=True)
for f in range(5):
    f_train = os.path.join(splits_dir, f"fold{f}_train.txt")
    f_val = os.path.join(splits_dir, f"fold{f}_val.txt")
    if os.path.exists(f_train) and os.path.exists(f_val):
        n_tr = len([l for l in open(f_train) if l.strip()])
        n_va = len([l for l in open(f_val) if l.strip()])
        print(f"  [+] Fold {f}: {n_tr} train cases | {n_va} val cases", flush=True)
print("[OK] Dữ liệu đã sẵn sàng!", flush=True)

### ⚡ Bước 4: Kiểm tra Cấu hình Phần cứng Dual GPU (2x T4)

In [ ]:
import torch

print("=" * 60, flush=True)
assert torch.cuda.is_available(), "[CẢNH BÁO] Chưa bật GPU! Hãy chọn GPU T4 x 2 ở mục Settings."
num_gpus = torch.cuda.device_count()
print(f"[GPU] Phát hiện {num_gpus} GPU khả dụng:", flush=True)
for i in range(num_gpus):
    name = torch.cuda.get_device_name(i)
    vram = round(torch.cuda.get_device_properties(i).total_memory / (1024**3), 2)
    print(f"  [+] GPU {i}: {name} | VRAM: {vram} GB", flush=True)

if num_gpus >= 2:
    print("[*] Đã sẵn sàng chế độ ĐA GPU (2x T4 - DDP via torchrun) để tăng tốc và cân bằng VRAM!", flush=True)
else:
    print("[*] Chế độ đơn GPU (1x T4 hoặc P100) sẵn sàng.", flush=True)
print("=" * 60, flush=True)

### 📊 Bước 5: Đăng nhập & Cấu hình Weights & Biases (W&B)
Theo dõi loss components (supervised, consistency, consistency_weight) và biểu đồ mDice trên W&B Cloud.

In [ ]:
import os
import wandb

# API Key W&B tích hợp sẵn cho 1 lần Run
WANDB_API_KEY = "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"
WANDB_PROJECT = "BHSD_MeanTeacher"

os.environ["WANDB_API_KEY"] = WANDB_API_KEY
os.environ["WANDB_PROJECT"] = WANDB_PROJECT

print("[*] Đang kết nối tới Weights & Biases...", flush=True)
wandb.login(key=WANDB_API_KEY, relogin=True)
print(f"[SUCCESS] Đăng nhập W&B thành công! Dữ liệu huấn luyện sẽ được lưu tại project: '{WANDB_PROJECT}'", flush=True)

### 🏋️‍♂️ Bước 6: Huấn luyện Mean Teacher End-to-End (2x T4 DDP)
- **Model**: `vnet_single` (Kiến trúc VNet chuẩn 1 decoder, tối ưu cho Mean Teacher).
- **Batch size**: 4 (mỗi GPU T4 xử lý 2 mẫu: 1 labeled, 1 unlabeled).
- **Consistency**: Weighted MSE (downweight background 0.1, confidence threshold 0.85, sigmoid rampup 40 epoch).

In [ ]:
# === HUẤN LUYỆN MEAN TEACHER TRÊN 2 GPU T4 VỚI FRAMEWORK MỚI ===
import os
import sys
import subprocess
import torch

# --- THÔNG SỐ THÍ NGHIỆM ---
FOLD_ID = 0                                                  # Chọn fold (0..4)
TASK = "multiclass"                                          # "multiclass" (6 lớp) hoặc "binary" (2 lớp)
MODEL_NAME = "vnet_single"                                   # "vnet_single", "unet3d"
METHOD_NAME = "mean_teacher"
EXP_NAME = f"BHSD_{TASK}_MeanTeacher_dual_gpu_fold{FOLD_ID}"  # Tên thí nghiệm độc lập
EXP_DIR = "/kaggle/working/experiments"
CONFIG_FILE = "configs/methods/mean_teacher.yaml"
MAX_ITER = 12000                                             # Tổng số iterations
VAL_INTERVAL = 500                                           # Đánh giá validation trượt 3D mỗi 500 bước
SAVE_INTERVAL = 50                                           # Lưu checkpoint_latest.pth mỗi 50 bước
BATCH_SIZE = 4                                               # Batch size tổng (2 GPU -> 2 sample/GPU)
LABELED_BS = 2                                               # 2 mẫu có nhãn trong batch

# Tự động phát hiện chế độ Đa GPU: Ưu tiên DDP nếu có >= 2 GPU
num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
use_ddp = (num_gpus >= 2)

base_args = [
    "--config", CONFIG_FILE,
    "--fold", str(FOLD_ID),
    "--task", TASK,
    "--model", MODEL_NAME,
    "--exp", EXP_NAME,
    "--exp_dir", EXP_DIR,
    "--resume",
    "--root_path", str(DATA_ROOT),
    "--batch_size", str(BATCH_SIZE),
    "--labeled_bs", str(LABELED_BS),
    "--max_iteration", str(MAX_ITER),
    "--val_interval", str(VAL_INTERVAL),
    "--save_interval", str(SAVE_INTERVAL),
    "--use_wandb",
    "--save_wandb_model",
    "--wandb_project", str(WANDB_PROJECT)
]

# Cấu hình Anti-Deadlock cho môi trường Đa GPU (2x T4):
import socket
def find_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(('', 0))
        return s.getsockname()[1]
master_port = str(find_free_port())

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["WANDB_API_KEY"] = WANDB_API_KEY
# Tránh deadlock do driver mạng ảo hóa / PCIe P2P trên Kaggle
env["NCCL_DEBUG"] = "WARN"
env["NCCL_IB_DISABLE"] = "1"
env["NCCL_P2P_DISABLE"] = "1"
env["TORCH_DISTRIBUTED_DEBUG"] = "DETAIL"

if use_ddp:
    print(f"[*] Phát hiện {num_gpus} GPU -> Khởi chạy chế độ DDP (torchrun) port={master_port}...", flush=True)
    train_cmd = [
        "torchrun",
        f"--nproc_per_node={num_gpus}",
        f"--master_port={master_port}",
        "train.py",
        "--ddp"
    ] + base_args
else:
    print("[*] Chế độ: Đơn GPU hoặc DataParallel (DP) qua python...", flush=True)
    train_cmd = [
        "python", "-u", "train.py",
        "--gpu", "auto"
    ] + base_args

print("=" * 75, flush=True)
print(f"[*] Khởi chạy lệnh:\n{' '.join(train_cmd)}", flush=True)
print(f"[*] Theo dõi biểu đồ W&B Cloud tại: https://wandb.ai/{WANDB_PROJECT}", flush=True)
print("=" * 75, flush=True)

# Stream log thời gian thực ra màn hình
process = subprocess.Popen(
    train_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env
)

for line in process.stdout:
    print(line, end="", flush=True)

process.wait()

print("\n" + "=" * 75, flush=True)
if process.returncode != 0:
    raise RuntimeError(f"[LỖI] Huấn luyện kết thúc với mã lỗi {process.returncode}!")
else:
    print(f"[SUCCESS] Huấn luyện hoàn tất thành công cho {EXP_NAME}!", flush=True)
print("=" * 75, flush=True)

### 🔬 Bước 7: Đánh giá Độc lập với Unified Benchmark (`evaluate.py`)
Nạp checkpoint tốt nhất (`best_model.pth` chứa trọng số của EMA Teacher) để tính toán điểm mDice, per-class Dice và đo lường FLOPs, Latency.

In [ ]:
import os
import subprocess

snapshot_dir = os.path.join(EXP_DIR, EXP_NAME, f"fold_{FOLD_ID}")
best_pth = os.path.join(snapshot_dir, "best_model.pth")
if not os.path.exists(best_pth):
    best_pth = os.path.join(snapshot_dir, "checkpoint_latest.pth")

if os.path.exists(best_pth):
    print(f"[✓] Tìm thấy best model checkpoint (EMA Teacher): {best_pth}", flush=True)
    eval_cmd = [
        "python", "evaluate.py",
        "--checkpoint", best_pth,
        "--root_path", str(DATA_ROOT),
        "--task", TASK,
        "--model", MODEL_NAME,
        "--fold", str(FOLD_ID),
        "--stride_xy", "32",
        "--stride_z", "8",
        "--profile"
    ]
    print(f"[*] Chạy đánh giá:\n{' '.join(eval_cmd)}\n", flush=True)
    subprocess.run(eval_cmd, check=True)
else:
    print(f"[!] Chưa tìm thấy checkpoint tại {best_pth}. Hãy kiểm tra lại thư mục snapshot.", flush=True)

### 📦 Bước 8: Đóng gói Kết quả Checkpoint & Logs
Nén toàn bộ kết quả mô hình thành file `.tar.gz` để tải về trực tiếp từ mục **Output** của Kaggle.

In [ ]:
import os

OUTPUT_ARCHIVE = f"/kaggle/working/{EXP_NAME}_results.tar.gz"

if os.path.exists(EXP_DIR):
    print(f"[*] Đang nén thư mục kết quả {EXP_DIR}...", flush=True)
    !tar -czf {OUTPUT_ARCHIVE} -C {EXP_DIR} .
    size_mb = round(os.path.getsize(OUTPUT_ARCHIVE) / (1024**2), 2)
    print(f"\n[✓] Đã tạo thành công file lưu trữ: {OUTPUT_ARCHIVE} ({size_mb} MB)", flush=True)
    print("[*] File đã sẵn sàng tải về tại mục Output bên phải màn hình.", flush=True)
    print(f"[*] Xem lại toàn bộ biểu đồ và so sánh: https://wandb.ai/{WANDB_PROJECT}", flush=True)
else:
    print(f"[!] Thư mục {EXP_DIR} không tồn tại.", flush=True)